# Window Size & Stride Search


## Config

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import copy
import time
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, r2_score

work_dir = os.getcwd()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"work_dir : {work_dir}")
print(f"device   : {DEVICE}")

# Paths
PROC_DIR = os.path.join(work_dir, "data_processed")

# Search grids
FACTOR = 3  # Factor = 3 for 1/3 downsampling, 1 for no downsampling
GRU_WINDOWS = [
    1000 // FACTOR,
    2000 // FACTOR,
    3000 // FACTOR,
    4000 // FACTOR,
    5000 // FACTOR,
    6000 // FACTOR,
    7000 // FACTOR,
    8000 // FACTOR,
    9000 // FACTOR,
    10000 // FACTOR,
]
CNN_WINDOWS = [
    1000 // FACTOR,
    2000 // FACTOR,
    3000 // FACTOR,
    4000 // FACTOR,
    5000 // FACTOR,
    6000 // FACTOR,
    7000 // FACTOR,
    8000 // FACTOR,
    9000 // FACTOR,
    10000 // FACTOR,
]

GRU_BEST_WINDOW = 3000 // FACTOR
CNN_BEST_WINDOW = 5000 // FACTOR

STRIDE_FRACTIONS = [1 / 16, 1 / 8, 1 / 4, 1 / 2, 1]  # stride = window * fraction

# Training
TOTAL_GRAD_STEPS = 25_000  # same for every run
BATCH_SIZE = 128
LR = 1e-3
HUBER_DELTA = 1.5  # % SoH — loss transitions from L2 to L1 at this error

# Early stopping
# Each run saves the best-val-loss checkpoint and stops if val does not
# improve for PATIENCE epochs.  Test evaluation always uses that checkpoint.
PATIENCE = 20

# Split config
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

# GRU model hyperparams
GRU_HIDDEN = 64
GRU_LAYERS = 2
GRU_DROPOUT = 0.05

# CNN+GRU model hyperparams
CNN_BASE_CH = 32
CNN_GRU_HIDDEN = 128
CNN_GRU_LAYERS = 2
CNN_DROPOUT = 0.05

print("Config loaded.")

## Helper Functions

In [14]:
import random as _rnd


def get_battery_id(fname):
    parts = fname.split("_")
    for p in parts:
        if p.startswith("P") and p[1:].isdigit():
            return p
    return parts[1]


def compute_derivatives(window):
    dv = np.gradient(window[:, 0])  # voltage is col 0 after dropping time
    di = np.gradient(window[:, 1])  # current is col 1
    return dv, di


def create_windows(x, window_size, stride, add_derivatives=True):
    X, y = [], []
    n = x.shape[0]
    for i in range(0, n - window_size + 1, stride):
        window = x[i : i + window_size]
        features = window[:, 1:-1].copy()  # (T, base_feats)

        if add_derivatives:
            dv, di = compute_derivatives(features)
            features = np.concatenate(
                [features, dv[:, None], di[:, None]], axis=1
            )  # (T, base_feats + 2)

        X.append(features)
        y.append(window[-1, -1])  # last SoH label in the window
        # y.append(window[:, -1].mean())          # mean SoH label
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def build_dataset(window_size, stride, seed=SEED):
    proc_files = sorted([f for f in os.listdir(PROC_DIR) if f.endswith(".npy")])

    battery_dict = {}
    for f in proc_files:
        bid = get_battery_id(f)
        battery_dict.setdefault(bid, []).append(f)

    batteries = list(battery_dict.keys())
    _rnd.seed(seed)
    _rnd.shuffle(batteries)
    n = len(batteries)
    train_b = batteries[: int(TRAIN_RATIO * n)]
    val_b = batteries[int(TRAIN_RATIO * n) : int((TRAIN_RATIO + VAL_RATIO) * n)]
    test_b = batteries[int((TRAIN_RATIO + VAL_RATIO) * n) :]

    def load_split(blist):
        Xs, ys = [], []
        for b in blist:
            for f in battery_dict[b]:
                x = np.load(os.path.join(PROC_DIR, f))
                if x.shape[0] < window_size:
                    continue
                Xw, yw = create_windows(x, window_size, stride)
                if len(Xw):
                    Xs.append(Xw)
                    ys.append(yw)
        if not Xs:
            return np.zeros((0, window_size, 5), np.float32), np.zeros(0, np.float32)
        return np.concatenate(Xs), np.concatenate(ys)

    X_train, y_train = load_split(train_b)
    X_val, y_val = load_split(val_b)
    X_test, y_test = load_split(test_b)

    # Normalize features (fit on train only)
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True) + 1e-10
    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std

    return (X_train, y_train, X_val, y_val, X_test, y_test)


def make_loader(X, y, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    return DataLoader(
        TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False
    )


def active_window_fraction(X, threshold=0.2):
    # Current is channel index 1 (after voltage)
    current_ranges = X[:, :, 1].max(axis=1) - X[:, :, 1].min(axis=1)
    return (current_ranges > threshold).mean()


def steps_per_epoch(n_samples):
    return max(1, int(np.ceil(n_samples / BATCH_SIZE)))


def epochs_for_budget(n_samples):
    return max(1, int(np.ceil(TOTAL_GRAD_STEPS / steps_per_epoch(n_samples))))


print("Helpers defined.")

## Model Definitions

In [15]:
# Attention module
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, gru_out):
        score = self.v(torch.tanh(self.attn(gru_out)))
        weight = torch.softmax(score, dim=1)
        context = (weight * gru_out).sum(dim=1)
        return context


# GRU with attention
class GRUModel(nn.Module):
    def __init__(self, input_size=5):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=GRU_HIDDEN,
            num_layers=GRU_LAYERS,
            batch_first=True,
            dropout=GRU_DROPOUT if GRU_LAYERS > 1 else 0.0,
        )
        self.attention = Attention(GRU_HIDDEN)
        self.dropout = nn.Dropout(GRU_DROPOUT)
        self.fc = nn.Linear(GRU_HIDDEN, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        ctx = self.attention(out)
        return self.fc(self.dropout(ctx)).squeeze(-1)


# CNN + GRU with attention
class CNN_GRU(nn.Module):
    def __init__(self, input_channels=5):
        super().__init__()
        bc = CNN_BASE_CH
        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, bc, 7, padding=3),
            nn.BatchNorm1d(bc),
            nn.ReLU(),
            nn.Conv1d(bc, bc * 2, 5, padding=2),
            nn.BatchNorm1d(bc * 2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(bc * 2, bc * 2, 3, padding=1),
            nn.BatchNorm1d(bc * 2),
            nn.ReLU(),
        )
        self.gru = nn.GRU(
            input_size=bc * 2,
            hidden_size=CNN_GRU_HIDDEN,
            num_layers=CNN_GRU_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=CNN_DROPOUT if CNN_GRU_LAYERS > 1 else 0.0,
        )

        self.attention = Attention(CNN_GRU_HIDDEN * 2)
        self.head = nn.Sequential(
            nn.Dropout(CNN_DROPOUT),
            nn.Linear(CNN_GRU_HIDDEN * 2, CNN_GRU_HIDDEN),
            nn.ReLU(),
            nn.Linear(CNN_GRU_HIDDEN, 1),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        out, _ = self.gru(x)
        ctx = self.attention(out)
        return self.head(ctx).squeeze(-1)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"GRU params    : {count_params(GRUModel()):,}")
print(f"CNN+GRU params: {count_params(CNN_GRU()):,}")

## Training Function

In [16]:
def train_one_config(model_class, window_size, stride, label=""):
    print(f"\n{'─' * 60}")
    print(f"  {label}  |  window={window_size}  stride={stride}")
    print(f"{'─' * 60}")

    # Data
    t0 = time.time()
    data = build_dataset(window_size, stride)
    X_train, y_train, X_val, y_val, X_test, y_test = data

    n_train, n_val, n_test = len(y_train), len(y_val), len(y_test)
    act_frac = active_window_fraction(X_train)

    n_epochs = epochs_for_budget(n_train)
    actual_steps = n_epochs * steps_per_epoch(n_train)

    print(
        f"  train={n_train:,}  val={n_val:,}  test={n_test:,}  "
        f"active={act_frac:.0%}  [{time.time() - t0:.1f}s]"
    )
    print(
        f"  epochs={n_epochs}  steps/epoch={steps_per_epoch(n_train)}  "
        f"total_steps≈{actual_steps:,}  (target={TOTAL_GRAD_STEPS:,})"
    )

    if n_train < 50 or n_val < 10:
        print("   Too few windows — skipping.")
        return None

    train_loader = make_loader(X_train, y_train)
    val_loader = make_loader(X_val, y_val, shuffle=False)
    test_loader = make_loader(X_test, y_test, shuffle=False)

    # Model
    input_channels = X_train.shape[2]
    model = model_class(input_channels).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.HuberLoss(delta=HUBER_DELTA)  # robust to outlier windows
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_val_rmse = float("inf")
    best_epoch = 0
    best_state = copy.deepcopy(model.state_dict())
    patience_count = 0

    # Training loop
    for epoch in range(1, n_epochs + 1):
        # Train
        model.train()
        bl = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            bl.append(loss.item())
        train_loss = np.mean(bl)

        # Validate
        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl.append(criterion(model(xb), yb).item())
        val_loss = np.mean(vl)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step()

        # Checkpoint: save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_rmse = np.sqrt(val_loss)
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

        if epoch % max(1, n_epochs // 10) == 0 or epoch == 1:
            gap = val_loss - train_loss
            print(
                f"  Ep {epoch:03d}/{n_epochs} | "
                f"train={train_loss:.4f}  val={val_loss:.4f}  "
                f"gap={gap:+.4f}  "
                f"{'← best' if epoch == best_epoch else ''}"
            )

        # Early stopping
        if patience_count >= PATIENCE:
            print(
                f"  Early stop at epoch {epoch} "
                f"(no improvement for {PATIENCE} epochs). "
                f"Best epoch was {best_epoch}."
            )
            break

    # Test evaluation
    model.load_state_dict(best_state)
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
            trues.append(yb.numpy())
    y_pred = np.concatenate(preds)
    y_true = np.concatenate(trues)

    test_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    test_r2 = r2_score(y_true, y_pred)

    # Overfitting gap at best epoch
    overfit_gap = val_losses[best_epoch - 1] - train_losses[best_epoch - 1]

    print(f"  Best epoch : {best_epoch}/{epoch}")
    print(f"  Test RMSE  : {test_rmse:.4f}    R² : {test_r2:.4f}")
    print(f"  Overfit gap: {overfit_gap:+.4f} (val - train loss at best epoch)")

    return {
        "window": window_size,
        "stride": stride,
        "n_train": n_train,
        "n_val": n_val,
        "n_test": n_test,
        "active_frac": float(act_frac),
        "n_epochs_run": epoch,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_rmse": best_val_rmse,
        "overfit_gap": overfit_gap,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "train_curve": train_losses,
        "val_curve": val_losses,
        "y_test": y_true,
        "y_pred": y_pred,
        "total_steps": epoch * steps_per_epoch(n_train),
    }


print("Training function defined.")


# STEP 1 — Sweep Window Sizes

In [6]:
gru_window_results = []
for W in GRU_WINDOWS:
    stride = max(1, W // 4)
    res = train_one_config(GRUModel, W, stride, label="GRU")
    if res is not None:
        gru_window_results.append(res)

print("\nGRU window sweep complete.")

In [ ]:
cnn_window_results = []
for W in CNN_WINDOWS:
    stride = max(1, W // 4)
    res = train_one_config(CNN_GRU, W, stride, label="CNN+GRU")
    if res is not None:
        cnn_window_results.append(res)

print("\nCNN+GRU window sweep complete.")

## Step 1 — Results Plot

In [ ]:
def plot_window_sweep(results, model_name, color):
    if not results:
        print(f"No results for {model_name}")
        return None

    windows = [r["window"] for r in results]
    val_rmses = [r["best_val_rmse"] for r in results]
    test_rmses = [r["test_rmse"] for r in results]
    test_r2s = [r["test_r2"] for r in results]
    n_trains = [r["n_train"] for r in results]
    act_fracs = [r["active_frac"] for r in results]
    overfit_gaps = [r["overfit_gap"] for r in results]
    total_steps = [r["total_steps"] for r in results]
    best_epochs = [r["best_epoch"] for r in results]
    epochs_run = [r["n_epochs_run"] for r in results]

    best_idx = int(np.argmin(val_rmses))

    fig = plt.figure(figsize=(20, 12))
    fig.suptitle(
        f"{model_name} — Window Size Sweep  "
        f"(stride=window//4, equalized gradient budget)",
        fontsize=14,
        fontweight="bold",
    )
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

    # Val RMSE
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(windows, val_rmses, "o-", color=color, lw=2, ms=8)
    ax.scatter(
        [windows[best_idx]],
        [val_rmses[best_idx]],
        s=150,
        color="red",
        zorder=5,
        label=f"best=W{windows[best_idx]}",
    )
    ax.set_title("Val RMSE vs Window")
    ax.set_xlabel("Window size")
    ax.set_ylabel("Val RMSE")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Test RMSE
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(windows, test_rmses, "s-", color=color, lw=2, ms=8)
    ax.set_title("Test RMSE vs Window")
    ax.set_xlabel("Window size")
    ax.set_ylabel("Test RMSE")
    ax.grid(alpha=0.3)

    # Test R²
    ax = fig.add_subplot(gs[0, 2])
    ax.plot(windows, test_r2s, "^-", color=color, lw=2, ms=8)
    ax.set_title("Test R² vs Window")
    ax.set_xlabel("Window size")
    ax.set_ylabel("R²")
    ax.grid(alpha=0.3)

    # Overfitting gap — new panel
    ax = fig.add_subplot(gs[0, 3])
    bar_colors = ["tomato" if g > 0.01 else "steelblue" for g in overfit_gaps]
    ax.bar(range(len(windows)), overfit_gaps, color=bar_colors, alpha=0.8)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(0.01, color="red", ls="--", lw=1, label="overfit threshold")
    ax.set_xticks(range(len(windows)))
    ax.set_xticklabels(windows)
    ax.set_title("Overfit Gap @ Best Epoch\n(val−train loss, lower=better)")
    ax.set_xlabel("Window size")
    ax.set_ylabel("Gap")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3, axis="y")

    # Training curves with best-epoch marker
    ax = fig.add_subplot(gs[1, :2])
    cmap = plt.cm.get_cmap("Blues", len(results) + 2)
    for i, r in enumerate(results):
        c = cmap(i + 2)
        ep = list(range(1, len(r["train_curve"]) + 1))
        ax.plot(
            ep, r["train_curve"], color=c, ls="-", lw=1.2, label=f"W={r['window']} tr"
        )
        ax.plot(
            ep, r["val_curve"], color=c, ls="--", lw=1.2, label=f"W={r['window']} val"
        )
        ax.axvline(r["best_epoch"], color=c, ls=":", lw=1.0, alpha=0.7)
    ax.set_title("Training Curves  (dotted line = best epoch checkpoint)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Huber Loss")
    ax.legend(fontsize=6, ncol=2)
    ax.grid(alpha=0.3)

    # Gradient steps actually used (fairness check)
    ax = fig.add_subplot(gs[1, 2])
    ax.bar(range(len(windows)), total_steps, color=color, alpha=0.7)
    ax.axhline(
        TOTAL_GRAD_STEPS, color="red", ls="--", label=f"target={TOTAL_GRAD_STEPS:,}"
    )
    ax.set_xticks(range(len(windows)))
    ax.set_xticklabels(windows)
    ax.set_title("Total Gradient Steps\n(should be ≈ equal)")
    ax.set_xlabel("Window size")
    ax.set_ylabel("Steps")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3, axis="y")

    # Best-epoch vs total-epochs ratio
    ax = fig.add_subplot(gs[1, 3])
    ratios = [b / e for b, e in zip(best_epochs, epochs_run)]
    ax.bar(range(len(windows)), ratios, color=color, alpha=0.7)
    ax.axhline(1.0, color="green", ls="--", lw=1, label="no early stop")
    ax.set_xticks(range(len(windows)))
    ax.set_xticklabels(windows)
    ax.set_title("Best epoch / Total epochs\n(low = early stopped = overfit)")
    ax.set_xlabel("Window size")
    ax.set_ylabel("Ratio")
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3, axis="y")

    # Active window fraction
    ax = fig.add_subplot(gs[2, 0])
    ax.bar(range(len(windows)), [f * 100 for f in act_fracs], color=color, alpha=0.7)
    ax.axhline(40, color="red", ls="--", label="40% threshold")
    ax.set_xticks(range(len(windows)))
    ax.set_xticklabels(windows)
    ax.set_title("Active Window Fraction")
    ax.set_xlabel("Window size")
    ax.set_ylabel("Active (%)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis="y")

    # Training windows count
    ax = fig.add_subplot(gs[2, 1])
    bars = ax.bar(range(len(windows)), n_trains, color=color, alpha=0.7)
    ax.set_xticks(range(len(windows)))
    ax.set_xticklabels(windows)
    ax.set_title("Training Windows per Config")
    ax.set_xlabel("Window size")
    ax.set_ylabel("# windows")
    for bar, v in zip(bars, n_trains):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(n_trains) * 0.01,
            f"{v:,}",
            ha="center",
            fontsize=7,
        )
    ax.grid(alpha=0.3, axis="y")

    # Best prediction scatter
    ax = fig.add_subplot(gs[2, 2:])
    best = results[best_idx]
    ax.scatter(best["y_test"], best["y_pred"], alpha=0.3, s=10, color=color)
    lo = min(best["y_test"].min(), best["y_pred"].min())
    hi = max(best["y_test"].max(), best["y_pred"].max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_title(
        f"Best config: W={windows[best_idx]}  "
        f"(best epoch={best['best_epoch']}/{best['n_epochs_run']})\n"
        f"RMSE={best['test_rmse']:.3f}  R²={best['test_r2']:.4f}"
    )
    ax.set_xlabel("True SoH")
    ax.set_ylabel("Predicted SoH")
    ax.grid(alpha=0.3)

    os.makedirs(os.path.join(work_dir, "figures"), exist_ok=True)
    plt.savefig(
        os.path.join(
            work_dir, f"figures/{model_name.replace('+', '_')}_window_sweep.png"
        ),
        dpi=120,
        bbox_inches="tight",
    )
    plt.show()

    print(f"\n{model_name} — WINDOW SWEEP SUMMARY:")
    print(
        f"{'Window':>8} {'Stride':>8} {'N_train':>9} {'Active':>8} "
        f"{'Steps':>8} {'BestEp':>7} {'ValRMSE':>9} {'TestRMSE':>10} "
        f"{'R²':>8} {'OvfGap':>9}"
    )
    print("-" * 90)
    for r in results:
        marker = " ← best" if r["window"] == windows[best_idx] else ""
        print(
            f"{r['window']:>8} {r['stride']:>8} {r['n_train']:>9,} "
            f"{r['active_frac']:>7.0%} {r['total_steps']:>8,} "
            f"{r['best_epoch']:>7}/{r['n_epochs_run']:<4} "
            f"{r['best_val_rmse']:>9.4f} {r['test_rmse']:>10.4f} "
            f"{r['test_r2']:>8.4f} {r['overfit_gap']:>+9.4f}{marker}"
        )

    return windows[best_idx]


os.makedirs(os.path.join(work_dir, "figures"), exist_ok=True)
gru_best_window = plot_window_sweep(gru_window_results, "GRU", "steelblue")
cnn_best_window = plot_window_sweep(cnn_window_results, "CNN+GRU", "darkorange")

if GRU_BEST_WINDOW is not None:
    gru_best_window = GRU_BEST_WINDOW
if CNN_BEST_WINDOW is not None:
    cnn_best_window = CNN_BEST_WINDOW

print(f"\n→ GRU     best window : {gru_best_window}")
print(f"→ CNN+GRU best window : {cnn_best_window}")


# STEP 2 — Sweep Stride


In [19]:
gru_stride_results = []
for frac in STRIDE_FRACTIONS:
    stride = max(1, int(GRU_BEST_WINDOW * frac))
    res = train_one_config(GRUModel, GRU_BEST_WINDOW, stride, label="GRU")
    if res is not None:
        gru_stride_results.append(res)

print("\nGRU stride sweep complete.")

In [21]:
cnn_stride_results = []
for frac in STRIDE_FRACTIONS:
    stride = max(1, int(CNN_BEST_WINDOW * frac))
    res = train_one_config(CNN_GRU, CNN_BEST_WINDOW, stride, label="CNN+GRU")
    if res is not None:
        cnn_stride_results.append(res)

print("\nCNN+GRU stride sweep complete.")

## Step 2 — Results Plot

In [23]:
def plot_stride_sweep(results, model_name, best_window, color):
    if not results:
        print(f"No results for {model_name}")
        return None

    strides = [r["stride"] for r in results]
    val_rmses = [r["best_val_rmse"] for r in results]
    test_rmses = [r["test_rmse"] for r in results]
    test_r2s = [r["test_r2"] for r in results]
    n_trains = [r["n_train"] for r in results]
    act_fracs = [r["active_frac"] for r in results]
    overfit_gaps = [r["overfit_gap"] for r in results]
    total_steps = [r["total_steps"] for r in results]
    best_epochs = [r["best_epoch"] for r in results]
    epochs_run = [r["n_epochs_run"] for r in results]

    best_idx = int(np.argmin(val_rmses))

    fig = plt.figure(figsize=(20, 12))
    fig.suptitle(
        f"{model_name} — Stride Sweep  (window={best_window}, "
        f"equalized gradient budget)",
        fontsize=14,
        fontweight="bold",
    )
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

    # Val RMSE
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(strides, val_rmses, "o-", color=color, lw=2, ms=8)
    ax.scatter(
        [strides[best_idx]],
        [val_rmses[best_idx]],
        s=150,
        color="red",
        zorder=5,
        label=f"best=s{strides[best_idx]}",
    )
    ax.set_title("Val RMSE vs Stride")
    ax.set_xlabel("Stride")
    ax.set_ylabel("Val RMSE")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Test RMSE
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(strides, test_rmses, "s-", color=color, lw=2, ms=8)
    ax.set_title("Test RMSE vs Stride")
    ax.set_xlabel("Stride")
    ax.set_ylabel("Test RMSE")
    ax.grid(alpha=0.3)

    # R²
    ax = fig.add_subplot(gs[0, 2])
    ax.plot(strides, test_r2s, "^-", color=color, lw=2, ms=8)
    ax.set_title("Test R² vs Stride")
    ax.set_xlabel("Stride")
    ax.set_ylabel("R²")
    ax.grid(alpha=0.3)

    # Overfitting gap
    ax = fig.add_subplot(gs[0, 3])
    bar_colors = ["tomato" if g > 0.01 else "steelblue" for g in overfit_gaps]
    ax.bar(range(len(strides)), overfit_gaps, color=bar_colors, alpha=0.8)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(0.01, color="red", ls="--", lw=1, label="overfit threshold")
    ax.set_xticks(range(len(strides)))
    ax.set_xticklabels(strides)
    ax.set_title("Overfit Gap @ Best Epoch")
    ax.set_xlabel("Stride")
    ax.set_ylabel("Gap")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3, axis="y")

    # Training curves with best-epoch markers
    ax = fig.add_subplot(gs[1, :2])
    cmap = plt.cm.get_cmap(
        "Oranges" if color == "darkorange" else "Blues", len(results) + 2
    )
    for i, r in enumerate(results):
        c = cmap(i + 2)
        ep = list(range(1, len(r["train_curve"]) + 1))
        ax.plot(
            ep, r["train_curve"], color=c, ls="-", lw=1.2, label=f"s={r['stride']} tr"
        )
        ax.plot(
            ep, r["val_curve"], color=c, ls="--", lw=1.2, label=f"s={r['stride']} val"
        )
        ax.axvline(r["best_epoch"], color=c, ls=":", lw=1.0, alpha=0.7)
    ax.set_title("Training Curves  (dotted = best epoch checkpoint)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Huber Loss")
    ax.legend(fontsize=6, ncol=2)
    ax.grid(alpha=0.3)

    # RMSE vs n_train elbow
    ax = fig.add_subplot(gs[1, 2])
    ax.plot(n_trains, val_rmses, "o-", color=color, lw=2, ms=8)
    for n, v, s in zip(n_trains, val_rmses, strides):
        ax.annotate(
            f"s={s}",
            (n, v),
            textcoords="offset points",
            xytext=(0, 8),
            fontsize=7,
            ha="center",
        )
    ax.set_title("Val RMSE vs # Windows\n(elbow = diminishing returns)")
    ax.set_xlabel("# training windows")
    ax.set_ylabel("Val RMSE")
    ax.grid(alpha=0.3)

    # Best-epoch ratio
    ax = fig.add_subplot(gs[1, 3])
    ratios = [b / e for b, e in zip(best_epochs, epochs_run)]
    ax.bar(range(len(strides)), ratios, color=color, alpha=0.7)
    ax.axhline(1.0, color="green", ls="--", lw=1, label="no early stop")
    ax.set_xticks(range(len(strides)))
    ax.set_xticklabels(strides)
    ax.set_title("Best epoch / Total epochs")
    ax.set_xlabel("Stride")
    ax.set_ylabel("Ratio")
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3, axis="y")

    # Active fraction
    ax = fig.add_subplot(gs[2, 0])
    ax.bar(range(len(strides)), [f * 100 for f in act_fracs], color=color, alpha=0.7)
    ax.axhline(40, color="red", ls="--", label="40% threshold")
    ax.set_xticks(range(len(strides)))
    ax.set_xticklabels(strides)
    ax.set_title("Active Window Fraction per Stride")
    ax.set_xlabel("Stride")
    ax.set_ylabel("Active (%)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis="y")

    # Gradient steps
    ax = fig.add_subplot(gs[2, 1])
    ax.bar(range(len(strides)), total_steps, color=color, alpha=0.7)
    ax.axhline(
        TOTAL_GRAD_STEPS, color="red", ls="--", label=f"target={TOTAL_GRAD_STEPS:,}"
    )
    ax.set_xticks(range(len(strides)))
    ax.set_xticklabels(strides)
    ax.set_title("Total Gradient Steps")
    ax.set_xlabel("Stride")
    ax.set_ylabel("Steps")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3, axis="y")

    # Best prediction scatter
    ax = fig.add_subplot(gs[2, 2:])
    best = results[best_idx]
    ax.scatter(best["y_test"], best["y_pred"], alpha=0.3, s=10, color=color)
    lo = min(best["y_test"].min(), best["y_pred"].min())
    hi = max(best["y_test"].max(), best["y_pred"].max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_title(
        f"Best config: stride={strides[best_idx]}  "
        f"(best epoch={best['best_epoch']}/{best['n_epochs_run']})\n"
        f"RMSE={best['test_rmse']:.3f}  R²={best['test_r2']:.4f}"
    )
    ax.set_xlabel("True SoH")
    ax.set_ylabel("Predicted SoH")
    ax.grid(alpha=0.3)

    plt.savefig(
        os.path.join(
            work_dir, f"figures/{model_name.replace('+', '_')}_stride_sweep.png"
        ),
        dpi=120,
        bbox_inches="tight",
    )
    plt.show()

    print(f"\n{model_name} — STRIDE SWEEP SUMMARY (window={best_window}):")
    print(
        f"{'Stride':>8} {'N_train':>9} {'Active':>8} {'Steps':>8} "
        f"{'BestEp':>7} {'ValRMSE':>9} {'TestRMSE':>10} "
        f"{'R²':>8} {'OvfGap':>9}"
    )
    print("-" * 84)
    for r in results:
        marker = " ← best" if r["stride"] == strides[best_idx] else ""
        print(
            f"{r['stride']:>8} {r['n_train']:>9,} "
            f"{r['active_frac']:>7.0%} {r['total_steps']:>8,} "
            f"{r['best_epoch']:>7}/{r['n_epochs_run']:<4} "
            f"{r['best_val_rmse']:>9.4f} {r['test_rmse']:>10.4f} "
            f"{r['test_r2']:>8.4f} {r['overfit_gap']:>+9.4f}{marker}"
        )

    return strides[best_idx]


gru_best_stride = plot_stride_sweep(
    gru_stride_results, "GRU", GRU_BEST_WINDOW, "steelblue"
)
cnn_best_stride = plot_stride_sweep(
    cnn_stride_results, "CNN+GRU", CNN_BEST_WINDOW, "darkorange"
)

# Final Recommendation

In [ ]:
print("=" * 60)
print("FINAL RECOMMENDED CONFIGS")
print("=" * 60)
print("  GRU / LSTM")
print(f"    window_size  = {gru_best_window}")
print(f"    fixed_stride = {gru_best_stride}")
print()
print("  CNN+GRU / CNN+LSTM / 1D-CNN")
print(f"    window_size  = {cnn_best_window}")
print(f"    fixed_stride = {cnn_best_stride}")
print("=" * 60)
print()
print("Copy these values into your preprocessing config cell")
print("and re-run 1_Preprocessing_Data.ipynb before training.")
print()
print("Note: results used best-epoch checkpoints throughout.")
print("The values above reflect the least-overfit model for each config.")